# Tome 10, Chapter 12: Random permutations and the Poisson–Dirichlet law (Python companion)

Exercises first, solutions at the end. We measure how fast the small cycles of a random permutation become independent Poisson variables, solve Dickman's delay equation, and check Billingsley's theorem on the two largest prime factors of the integers up to $10^7$. The computer science application is the analysis of **factoring by trial division** (Knuth and Trabb Pardo, 1976): how far must we divide before a number is completely factored? We also build Ewens permutations with the "Chinese restaurant" construction. Requires `numpy` and `sympy`.

In [ ]:
import math, random
import numpy as np
from sympy import isprime, primerange
rng = np.random.default_rng(120)
random.seed(120)

## Exercises

**P12.1 (★) Small cycles.** Using Theorem 12.4's proof, compute *exactly* the total variation distance between the law of $(C_1,C_2)$ for a uniform permutation of size $n$ and the law of independent Poisson variables with means $1$ and $\frac12$, for $n=4,6,8,10,15,20,30$. (You need $q_2(k)=\sum_{i\le k}g_i$ with $g(x)=e^{-x-x^2/2}$.) How fast does it decrease? Check $n=10$ by simulating $200\,000$ permutations.

**P12.2 (★) Dickman's function.** Solve $u\rho'(u)=-\rho(u-1)$, $\rho=1$ on $[0,1]$, numerically. Print $\rho(u)$ for $u=1,\dots,8$, next to $\frac1{\lfloor u\rfloor!}$ and $u^{-u}$, and compute the Golomb–Dickman constant $\lambda=\int_1^\infty\frac{1-\rho(u)}{u^2}\,du$.

**P12.3 (★★) Billingsley's theorem.** With a sieve, compute the largest and second largest *distinct* prime factors $p_1(n)\ge p_2(n)$ of every $n\le10^7$, and $Y_1=\frac{\log p_1}{\log x}$, $Y_2=\frac{\log p_2}{\log x}$ ($Y_2=0$ if $n$ has one prime factor), $x=10^7$. Compare the means of $Y_1$, $Y_2$, and the probabilities $P(Y_1\le a,Y_2\le b)$ for a few $(a,b)$, with the same quantities for $V_{(1)},V_{(2)}$ computed from $10^6$ broken sticks.

**P12.4 (★★) CS application: trial division.** To factor $n$, divide by $2,3,5,7,\dots$ up to a bound $B$, removing each prime factor found. What remains is $1$, a prime, or a composite with all factors $>B$. The method has *completely factored* $n$ when the cofactor is $1$ or prime, that is, when the second largest prime factor (with multiplicity) is at most $B$.

(a) For $2000$ random $n$ in $[10^{12},2\cdot10^{12}]$, and $B=n^{1/u}$ for $u=2,3,4,5$, measure the proportion completely factored, and compare with $P(V_{(2)}\le\frac1u)$ from the broken sticks.

(b) What bound $B$ do we need for the method to succeed $90\%$ of the time on 40-digit numbers? How does that compare with $\sqrt n$, the bound that always works?

**P12.5 (★★) Ewens permutations.** Implement the construction of Exercise 12.3 (the "Chinese restaurant process") for $\theta=\frac12,1,3$ and $n=1000$. Compare the mean number of cycles over $2000$ samples with $\sum_{i\le n}\frac\theta{\theta+i-1}$, and the mean relative length of the longest cycle with the case $\theta=1$. What does $\theta$ do to the broken stick? (For general $\theta$, the GEM sticks are broken with $U_i$ of density $\theta(1-u)^{\theta-1}$; check it by simulation.)

## Solutions

### P12.1
**Idea.** The proof of Theorem 12.4 gives the exact formula $P((C_1,C_2)=c)=P((Z_1,Z_2)=c)\,e^{H_2}q_2(n-c_1-2c_2)$, and $q_2$ is a partial sum of the coefficients of an entire function, whose coefficients decrease like $1/k!$ or so. The distance is the tail of those coefficients, hence superexponentially small.

In [ ]:
from fractions import Fraction
def g_coeffs(K):
    # coefficients of exp(-x - x^2/2): g' = (-1 - x) g
    g = [1.0] + [0.0]*K
    for k in range(K):
        # (k+1) g_{k+1} = -g_k - g_{k-1}
        g[k+1] = (-g[k] - (g[k-1] if k >= 1 else 0.0)) / (k+1)
    return g
K = 60
g = g_coeffs(K)
q2 = np.cumsum(g)
H2 = 1.5
def pz(c1, c2):
    return math.exp(-1)/math.factorial(c1) * math.exp(-0.5)*0.5**c2/math.factorial(c2)
def tv(n):
    s = 0.0; tot = 0.0
    for c1 in range(0, 40):
        for c2 in range(0, 30):
            m = c1 + 2*c2
            p = pz(c1, c2)
            pc = p*math.exp(H2)*q2[n-m] if m <= n else 0.0
            s += abs(pc - p); tot += pc
    return s/2, tot
for n in (4, 6, 8, 10, 15, 20, 30):
    d, tot = tv(n)
    print(f"n={n:2d}: TV = {d:.3e}   (sum of exact probabilities {tot:.6f})")

In [ ]:
def cycle_counts(p, b=2):
    seen = np.zeros(len(p), bool); c = [0]*(b+1)
    for i in range(len(p)):
        if not seen[i]:
            l = 0; j = i
            while not seen[j]:
                seen[j] = True; j = p[j]; l += 1
            if l <= b: c[l] += 1
    return c[1], c[2]
n, M = 10, 200000
from collections import Counter
cnt = Counter(cycle_counts(rng.permutation(n)) for _ in range(M))
emp = sum(abs(cnt.get((c1,c2),0)/M - pz(c1,c2)) for c1 in range(15) for c2 in range(10))/2
print(f"simulated TV for n=10: {emp:.4f} (exact {tv(10)[0]:.4f}; sampling noise adds about 0.003)")

**Write-up.** The exact distance falls from $0.46$ at $n=4$ to $0.012$ at $n=10$, $1.4\cdot10^{-4}$ at $n=15$, $7\cdot10^{-7}$ at $n=20$ and $3.5\cdot10^{-12}$ at $n=30$: faster than any geometric sequence, as Arratia and Tavaré proved. The total probability column is $1$, a check of the formula. The simulation agrees with the exact value up to its own noise. The Poisson model of the small cycles is excellent already for a deck of a few dozen cards.

### P12.2
**Idea.** On each interval $[k,k+1]$ the equation is an integral of known values on $[k-1,k]$: march forward with the trapezoidal rule on a fine grid.

In [ ]:
h = 1e-4; U = 12.0; N = int(round(U/h)) + 1; m = int(round(1/h))
rho = np.ones(N)
for i in range(m+1, N):
    u = i*h
    rho[i] = rho[i-1] - h*(rho[i-1-m]/(u-h) + rho[i-m]/u)/2
ug = np.arange(N)*h
R = lambda u: float(np.interp(u, ug, rho))
print(" u    rho(u)        1/floor(u)!   u^-u")
for u in range(1, 9):
    print(f"{u:2d}  {R(u):.6e}  {1/math.factorial(u):.3e}  {u**-u:.3e}")
sel = ug >= 1
lam = np.trapezoid((1 - rho[sel])/ug[sel]**2, ug[sel]) + (1 - R(U))/U
print("Golomb-Dickman lambda =", lam)

**Write-up.** $\rho(2)=1-\log2=0.3069$, $\rho(3)=0.04861$, $\rho(4)=0.00491$, $\rho(8)=3.2\cdot10^{-8}$. (The trapezoidal rule has an absolute error of order $10^{-9}$ here, which is why we stop at $u=8$: beyond, $\rho$ is smaller than the error. A relative-accuracy method integrates $\log\rho$ instead.) The bound $\frac1{\lfloor u\rfloor!}$ is far from sharp: at $u=8$ it is $800$ times too large. The rough guess $u^{-u}$ stays within a factor $2$ of $\rho$ in this range, a little below it for $u\le5$ and above it after: de Bruijn's $\rho(u)=u^{-u(1+o(1))}$ is a statement about the exponent only. The constant is $\lambda=0.62432998\ldots$, as in Corollary 12.12.

### P12.3
**Idea.** A sieve over primes in increasing order records the last prime that hits each $n$ (the largest factor) and the previous one (the second largest). For the broken sticks, $20$ breaks leave a remainder $W_{20}$ of mean $2^{-20}$, negligible.

In [ ]:
X = 10**7
isp = np.ones(X+1, bool); isp[:2] = False
for p in range(2, int(X**0.5)+1):
    if isp[p]: isp[p*p::p] = False
p1 = np.ones(X+1, np.int32); p2 = np.ones(X+1, np.int32)
for p in np.nonzero(isp)[0]:
    p2[p::p] = p1[p::p]; p1[p::p] = p
lx = math.log(X)
Y1 = np.log(p1[1:].astype(float))/lx; Y2 = np.log(p2[1:].astype(float))/lx

Ms = 10**6
Us = rng.random((Ms, 20)); W = np.cumprod(1 - Us, axis=1)
V = Us * np.hstack([np.ones((Ms,1)), W[:, :-1]])
V = np.hstack([V, W[:, -1:]])          # the tiny remainder as one last piece
V.sort(axis=1); V1, V2 = V[:, -1], V[:, -2]
print(f"means:  Y1 {Y1.mean():.4f}  V1 {V1.mean():.4f}   |  Y2 {Y2.mean():.4f}  V2 {V2.mean():.4f}")
for a, b in [(0.5, 0.2), (0.7, 0.3), (0.9, 0.1), (1.0, 0.25), (0.6, 0.6)]:
    print(f"P(<= {a}, <= {b}):  integers {np.mean((Y1<=a)&(Y2<=b)):.4f}   sticks {np.mean((V1<=a)&(V2<=b)):.4f}")

**Write-up.** The means are within $0.02$ of the limits ($\mathbb EV_{(1)}=0.6243$, $\mathbb EV_{(2)}=0.2096$), but the joint probabilities differ by up to $0.1$: Billingsley's theorem is visible at $x=10^7$, but the convergence is slow, with errors of order $\frac1{\log x}\approx0.06$. The integers lag behind the limit in the same direction as in Figure 12.2: at this size their prime factors are a little smaller than the Poisson–Dirichlet law predicts, so the events "both factors small" are more frequent.

### P12.4
**Idea.** Trial division up to $B$ succeeds when at most one prime factor exceeds $B$, so its success probability is the law of the *second* largest factor, $P(V_{(2)}\le\frac1u)$ in the limit. Note $V_{(2)}\le\frac12$ always, so $B=\sqrt n$ always works.

In [ ]:
def trial_division_success(ns, B):
    arr = np.array(ns, dtype=object)
    a = np.array(ns, dtype=np.int64)
    for p in primerange(2, int(B)+1):
        mask = a % p == 0
        while mask.any():
            a[mask] //= p
            mask = a % p == 0
    return np.array([x == 1 or isprime(int(x)) for x in a])

ns = [random.randrange(10**12, 2*10**12) for _ in range(2000)]
for u in (2, 3, 4, 5):
    B = (1.5e12)**(1/u)
    ok = trial_division_success(ns, B)
    print(f"u={u}: B={B:10.0f}  factored {ok.mean():.3f}   P(V2 <= 1/u) = {np.mean(V2 <= 1/u):.3f}")

In [ ]:
# (b) the bound that succeeds 90% of the time, in the limit
qs = np.quantile(V2, 0.9)
print(f"90% quantile of V2: {qs:.4f}, so B = n^{qs:.3f}")
print(f"for a 40-digit n: B is about 10^{40*qs:.1f}, against sqrt(n) = 10^20")

**Write-up.** (a) The measured success rates agree with $P(V_{(2)}\le\frac1u)$ to within $0.01$, a better agreement than for the joint law in P12.3. For $u=2$ the success is certain, since the second largest factor is at most $\sqrt n$. With $B=n^{1/3}$ trial division completely factors about $85\%$ of the numbers, with $B=n^{1/5}$ almost half. (b) In the limit, $90\%$ success needs $B=n^{0.36}$ (the printed quantile): for a 40-digit number, about $10^{14.4}$ trial divisions, against $10^{20}$ for certainty. The last $10\%$, the numbers with two large prime factors, are what the modern algorithms are for. This is the analysis of Knuth and Trabb Pardo (1976).

### P12.5
**Idea.** The construction is a sequence of independent choices; with a linked representation ("next element in the cycle") the insertion is $O(1)$.

In [ ]:
def ewens(n, theta):
    nxt = np.empty(n, dtype=np.int64)
    for i in range(n):
        if random.random() < theta/(theta + i):
            nxt[i] = i                      # new cycle (i)
        else:
            j = random.randrange(i)         # insert i right after j
            nxt[i] = nxt[j]; nxt[j] = i
    return nxt

n, S = 1000, 2000
for theta in (0.5, 1.0, 3.0):
    Ks, Lm = [], []
    for _ in range(S):
        p = ewens(n, theta)
        seen = np.zeros(n, bool); k = 0; best = 0
        for i in range(n):
            if not seen[i]:
                k += 1; l = 0; j = i
                while not seen[j]:
                    seen[j] = True; j = p[j]; l += 1
                best = max(best, l)
        Ks.append(k); Lm.append(best/n)
    exact = sum(theta/(theta + i) for i in range(n))
    Ub = 1 - rng.random((200000, 40))**(1/theta)      # density theta(1-u)^(theta-1)
    Wb = np.cumprod(1 - Ub, axis=1)
    Vb = Ub*np.hstack([np.ones((200000,1)), Wb[:, :-1]])
    print(f"theta={theta}: mean cycles {np.mean(Ks):.3f} (exact {exact:.3f});  mean longest/n {np.mean(Lm):.3f},  sticks {Vb.max(axis=1).mean():.3f}")

**Write-up.** The mean number of cycles matches $\sum_i\frac\theta{\theta+i-1}\approx\theta\log n$. A small $\theta$ makes few, long cycles (the longest holds most of the elements), a large $\theta$ many short ones. In the limit the cycle lengths are again a broken stick, but each break keeps a piece $U_i$ of density $\theta(1-u)^{\theta-1}$, of mean $\frac1{1+\theta}$: the law $\mathrm{PD}(\theta)$, which for $\theta\neq1$ appears in population genetics (Ewens, Kingman) and, for prime factors, in integers weighted by $\theta^{\omega(n)}$.